# Does the model predict by signer or source?

| EXP NO                               | Model           | Split            |
|--------------------------------------|-----------------|------------------|
| exp000, exp002                       | MViTv2_S        | asl2000_cutoff_9 |
| MViTv2_S_16x4_e_5mzfq88e_chained_s42 | MViTv2_S_16x4_e | asl2000          |

When the model is wrong, does it predict a gloss that the test video's *signer* (or *source*)
performs in training more often than chance? If so, it has partly learnt who is signing (or what
the recording looks like) rather than the sign. Asked by the thesis (Ch5 §5.5.5/§5.5.6 and the Ch6
Limitations) after a preliminary check on 2026-10-24; this notebook reproduces it and separates
the explanations:

1. **Reproduction** on the 0-based plain `asl2000` labels.
2. **Signer or source?** They're nested: most sources have several signers, but only 3 of 118
   signers appear in more than one source (`asl5200` is all signer 9).
3. **Appearance or sign similarity?** A wrong prediction on the signer's gloss could still be a
   similar-looking sign, and a signer performing the true gloss in training could help through the
   same sign form rather than their appearance.
4. **Which signers** drive it.
5. The same tests on **exp002** and the thesis's final asl2000 model (job 4).

The baseline shuffles the wrong predictions among the wrong test videos (2,000 permutations),
which keeps how often each gloss is predicted and each signer's gloss set. Signer, source and
gloss come from the plain `asl2000` split, without the three videos the 1-based `cutoff_9`
removed; the frame-range differences between the label versions don't affect them. Identical
copies are the exact duplicates of
[duplicate_accuracy.ipynb](./duplicate_accuracy.ipynb). Helpers:
[`signer_shortcut.py`](./signer_shortcut.py). Dataset fields and gotchas:
[WLASL_info.md](../../info/WLASL_info.md).

In [1]:
from collections import Counter, defaultdict
from typing import NamedTuple

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from IPython.display import display
from pydantic import TypeAdapter

from src.configs import get_class_list
from src.results import format_exp_label, get_out_stub, get_stash_path, load_json
from src.results.dataset_analysis.find_duplicates import (
    DEFAULT_REPORT,
    Report,
    load_split,
)
from src.results.satnac_2026.signer_shortcut import (
    CANDIDATE_FEATURES,
    PermutationResult,
    candidate_features,
    conditional_logit,
    cross_gloss_pairs,
    hits,
    other_signer_confusions,
    permutation_test,
    prediction_table,
    train_copy_glosses,
    train_gloss_sets,
)
from src.run_types import SPLIT_DIR, InstanceTopK

## Setup

In [2]:
class Run(NamedTuple):
    split: str
    model: str
    exp_no: str


run = Run('asl2000_cutoff_9', 'MViTv2_S', '000') # the run analysed in full
other_runs = [ # rerun on the headline tests at the end
    Run('asl2000_cutoff_9', 'MViTv2_S', '002'),
    Run('asl2000', 'MViTv2_S_16x4_e', 'MViTv2_S_16x4_e_5mzfq88e_chained_s42'), # thesis job 4's final model
]
n_perm = 2000
top_n = 5 # how many of other signers' predictions count as "confused with"

CUTOFF_REMOVED = {'18223', '59958', '15144'}
classes = get_class_list()
report = Report.model_validate_json(DEFAULT_REPORT.read_text())
members = load_split(SPLIT_DIR / 'asl2000.json')
signer_sets = train_gloss_sets(members.values(), 'signer_id', CUTOFF_REMOVED)
source_sets = train_gloss_sets(members.values(), 'source', CUTOFF_REMOVED)
train = [m for m in members.values() if m.set == 'train' and m.video_id not in CUTOFF_REMOVED]
train_counts = Counter(m.gloss for m in train)
copies = train_copy_glosses(report)
cross = cross_gloss_pairs(report)


def load_table(r: Run) -> pd.DataFrame:
    """The prediction table of one run's stashed test top-k."""
    stub = get_out_stub(r.split, r.model, format_exp_label(r.exp_no))
    predictions = TypeAdapter(list[InstanceTopK]).validate_python(
        load_json(get_stash_path(stub, base_name='instance_topk'))
    )
    df = prediction_table(predictions, members, classes, copies, top_n)
    df['others'] = other_signer_confusions(df)
    df['signer_true'] = hits(df, 'signer', signer_sets, 'gloss')
    df['source_true'] = hits(df, 'source', source_sets, 'gloss')
    return df


def wrong_of(df: pd.DataFrame, drop_copy_preds: bool = True) -> pd.DataFrame:
    """The wrong predictions, without those explained by an identical training copy by default."""
    keep = ~df['top1'] & ~(df['copy_pred'] & drop_copy_preds)
    return df[keep].reset_index(drop=True)


def results_table(rows: dict[str, PermutationResult]) -> pd.DataFrame:
    table = pd.DataFrame({k: v.as_row() for k, v in rows.items()}).T
    table['n'] = table['n'].astype(int)
    return table.round({'observed (%)': 1, 'shuffled (%)': 1, 'shuffled 97.5th pct (%)': 1, 'ratio': 2, 'p': 4})


df = load_table(run)
wrong = wrong_of(df)
print(f'{len(df)} test predictions, {int((~df["top1"]).sum())} wrong, '
      f'{int((~df["top1"] & df["copy_pred"]).sum())} of them the gloss of an identical training copy')

2879 test predictions, 1760 wrong, 147 of them the gloss of an identical training copy


## 1. Reproduction

How often a wrong prediction is a gloss that the same signer (or source) performs in training,
against the shuffled baseline. `p` is one-sided, from the permutations (its floor is 1/2,001).

In [3]:
subsets = {
    'all wrong': wrong_of(df, drop_copy_preds=False),
    'not explained by an identical copy': wrong,
}
results_table({
    f'{name}: same {key}': permutation_test(sub, key, sets, n_perm)
    for name, sub in subsets.items()
    for key, sets in [('signer', signer_sets), ('source', source_sets)]
})

,n,observed (%),shuffled (%),shuffled 97.5th pct (%),ratio,p
all wrong: same signer,1760,50.8,27.1,28.9,1.88,0.0005
all wrong: same source,1760,74.0,52.6,54.8,1.41,0.0005
not explained by an identical copy: same signer,1613,46.4,26.5,28.5,1.75,0.0005
not explained by an identical copy: same source,1613,71.7,51.7,54.0,1.39,0.0005


Top-1 accuracy (%) of test videos without an identical training copy, by whether the signer (or
source) performs the true gloss in training.

In [4]:
no_copy = df[~df['has_train_copy']]


def top1_by(table: pd.DataFrame, by: list[str]) -> pd.DataFrame:
    """Count and top-1 accuracy (%) of `table`'s rows, grouped by `by`."""
    out = table.groupby(by)['top1'].agg(n='size', top1='mean')
    out['top1'] = (out['top1'] * 100).round(2)
    return out


display(top1_by(no_copy, ['signer_true']), top1_by(no_copy, ['source_true']))
top1_by(no_copy, ['source_true', 'signer_true'])

,n,top1
signer_true,,
False,2428,38.26
True,295,61.36


,n,top1
source_true,,
False,2331,37.97
True,392,57.40


n   top1
source_true signer_true             
False       False        2320  37.80
            True           11  72.73
True        False         108  48.15
            True          284  60.92

## 2. Signer or source?

Signers are nested in sources, so two checks:

* **Shuffles within source.** Wrong predictions are only swapped between videos of the same
  source, so any excess left can't come from the source. (Shuffling within signer, the reverse,
  can't test source: a signer's videos almost always share one source.)
* **Conditional logit.** Each wrong prediction is a choice among the other 1,999 glosses, each
  described by whether the signer performs it in training, whether the source has it in training,
  how often the model predicts it on the *other* wrong videos, its training count, and two
  signer-independent signs that it looks like the true sign (section 3). The odds ratios are each
  feature's effect with the others held fixed.

In [5]:
results_table({
    'same signer, shuffled within source': permutation_test(wrong, 'signer', signer_sets, n_perm, strata='source'),
})

,n,observed (%),shuffled (%),shuffled 97.5th pct (%),ratio,p
"same signer, shuffled within source",1613,46.4,38.9,40.1,1.19,0.0005


In [6]:
features, chosen, available = candidate_features(
    wrong, list(wrong['others']), classes, signer_sets, source_sets, train_counts, cross
)
fit = conditional_logit(features, chosen, available, CANDIDATE_FEATURES)
fit.table().round(3)

,coef,se,odds ratio,OR 95% low,OR 95% high,z
same signer,0.900,0.078,2.460,2.112,2.866,11.571
same source,0.673,0.078,1.960,1.681,2.285,8.596
log predicted count,0.165,0.052,1.179,1.064,1.306,3.149
log train count,1.122,0.093,3.071,2.558,3.686,12.032
other-signer confusion,2.405,0.148,11.079,8.291,14.805,16.263
shares identical video,6.208,0.100,496.756,408.283,604.399,62.038


## 3. Appearance or sign similarity?

Two signer-independent signs that a predicted gloss looks like the true one:

* **Shares an identical video** with the true gloss somewhere in the dataset (the same sign filed
  under two glosses, e.g. `before`/`former`/`past` in
  [WLASL_info](../../info/WLASL_info.md#one-sign-several-glosses-eg-before--former--past)).
* **Other-signer confusion:** it is in the top-5 of another signer's test video of the true gloss.

WLASL's `variation_id` numbers forms *within* a gloss, so it can't say whether two glosses are
similar. If the signer effect worked through similar signs, wrong predictions landing on the
signer's glosses would be related to the true gloss more often than the others.

In [7]:
wrong['signer_hit'] = hits(wrong, 'signer', signer_sets)
wrong['shares video'] = [frozenset((a, b)) in cross for a, b in zip(wrong['gloss'], wrong['predicted'], strict=True)]
wrong['other-signer confusion'] = [p in o for p, o in zip(wrong['predicted'], wrong['others'], strict=True)]
wrong['related'] = wrong['shares video'] | wrong['other-signer confusion']
related = wrong.groupby('signer_hit')[['shares video', 'other-signer confusion', 'related']].mean() * 100
related.index = pd.Index(['prediction not in the signer\'s set', 'prediction in the signer\'s set'])
related.round(1)

,shares video,other-signer confusion,related
prediction not in the signer's set,18.9,9.4,23.6
prediction in the signer's set,15.4,11.1,22.4


The test again, on the wrong predictions unrelated to the true gloss by both measures.

In [8]:
unrelated = wrong[~wrong['related']].reset_index(drop=True)
results_table({
    'unrelated: same signer': permutation_test(unrelated, 'signer', signer_sets, n_perm),
    'unrelated: same signer, shuffled within source': permutation_test(
        unrelated, 'signer', signer_sets, n_perm, strata='source'
    ),
})

,n,observed (%),shuffled (%),shuffled 97.5th pct (%),ratio,p
unrelated: same signer,1241,46.8,25.9,28.0,1.81,0.0005
"unrelated: same signer, shuffled within source",1241,46.8,38.7,40.2,1.21,0.0005


### Does the signer help when right?

The accuracy gain from the signer performing the true gloss in training (section 1) could be
appearance or the same sign form. Split by whether the signer's training videos of the gloss
include the test video's variation (no identical copies):

In [9]:
signer_variations: defaultdict[tuple[int, str], set[int]] = defaultdict(set)
for m in train:
    signer_variations[(m.signer_id, m.gloss)].add(m.variation_id)

seen = no_copy[no_copy['signer_true']].copy()
seen['same variation'] = [
    v in signer_variations[(s, g)] for s, g, v in zip(seen['signer'], seen['gloss'], seen['variation'], strict=True)
]
top1_by(seen, ['same variation'])

,n,top1
same variation,,
False,42,38.10
True,253,65.22


And a logistic regression of top-1 on the same videos, with the gloss's training count and
training signers as covariates (the latter predicts accuracy in
[correlation_f1.ipynb](./correlation_f1.ipynb)).

In [10]:
train_signers: defaultdict[str, set[int]] = defaultdict(set)
for m in train:
    train_signers[m.gloss].add(m.signer_id)
acc = no_copy.assign(
    y=no_copy['top1'].astype(int),
    signer_true=no_copy['signer_true'].astype(int),
    source_true=no_copy['source_true'].astype(int),
    log_train=np.log(no_copy['gloss'].map(train_counts)),
    train_signers=no_copy['gloss'].map(lambda g: len(train_signers[g])),
)
logit = smf.logit('y ~ signer_true + source_true + log_train + train_signers', acc).fit(disp=0)
ci = np.exp(logit.conf_int())
pd.DataFrame({
    'odds ratio': np.exp(logit.params), 'OR 95% low': ci[0], 'OR 95% high': ci[1], 'p': logit.pvalues
}).round(4)

,odds ratio,OR 95% low,OR 95% high,p
Intercept,0.1070,0.0531,0.2157,0.0000
signer_true,1.7533,1.1343,2.7102,0.0115
source_true,1.1563,0.7873,1.6983,0.4591
log_train,0.9589,0.5099,1.8033,0.8964
train_signers,1.2944,1.1756,1.4253,0.0000


## 4. Which signers?

Each signer's wrong predictions on its own training glosses against the number expected under
the shuffle, sorted by the excess. `share` is the cumulative share of the total excess.

In [11]:
result = permutation_test(wrong, 'signer', signer_sets, n_perm)
per_signer = (
    wrong.assign(expected=result.expected)
    .groupby('signer')
    .agg(
        wrong=('video_id', 'size'),
        hits=('signer_hit', 'sum'),
        expected=('expected', 'sum'),
        sources=('source', lambda s: ', '.join(sorted(set(s)))),
    )
)
per_signer['train glosses'] = [len(signer_sets.get(s, ())) for s in per_signer.index]
per_signer['excess'] = per_signer['hits'] - per_signer['expected']
per_signer['ratio'] = per_signer['hits'] / per_signer['expected']
per_signer = per_signer.sort_values('excess', ascending=False)
per_signer['share'] = per_signer['excess'].cumsum() / per_signer['excess'].sum()
print(f'{len(per_signer)} signers with wrong predictions; excess {per_signer["excess"].sum():.0f} '
      f'of {int(per_signer["hits"].sum())} hits')
per_signer.head(15).round(2)

97 signers with wrong predictions; excess 321 of 749 hits


,wrong,hits,expected,sources,train glosses,excess,ratio,share
signer,,,,,,,,
9,383,260,163.28,asl5200,629,96.72,1.59,0.30
5,88,71,48.87,aslu,947,22.13,1.45,0.37
4,49,31,10.30,signschool,395,20.70,3.01,0.44
11,88,68,52.39,signingsavvy,1138,15.61,1.30,0.48
25,35,19,3.97,aslpro,205,15.03,4.79,0.53
14,59,36,22.34,handspeak,762,13.66,1.61,0.57
13,56,32,18.64,asldeafined,652,13.36,1.72,0.61
10,29,17,5.28,signingsavvy,373,11.72,3.22,0.65
12,33,28,18.27,aslsearch,1103,9.73,1.53,0.68


Signers with fewer hits than expected, among those expected to have at least one:

In [12]:
per_signer[(per_signer['excess'] < 0) & (per_signer['expected'] >= 1)].round(2)

,wrong,hits,expected,sources,train glosses,excess,ratio,share
signer,,,,,,,,
20,51,15,15.12,handspeak,611,-0.12,0.99,1.01
41,17,3,3.52,aslsearch,405,-0.52,0.85,1.00


The test without the largest signers:

In [13]:
results_table({
    f'without signer(s) {", ".join(map(str, drop))}': permutation_test(
        wrong[~wrong['signer'].isin(drop)].reset_index(drop=True), 'signer', signer_sets, n_perm
    )
    for drop in [[9], [9, 5, 11]]
})

,n,observed (%),shuffled (%),shuffled 97.5th pct (%),ratio,p
without signer(s) 9,1230,39.8,22.0,24.0,1.81,0.0005
"without signer(s) 9, 5, 11",1054,33.2,15.9,17.8,2.09,0.0005


## 5. Other runs

The headline tests for each run (wrong predictions not explained by an identical copy).
exp002 is the better asl2000 `MViTv2_S` run (same `data_info.json` as exp000; its config isn't
in the repo). Its test top-k was stashed on 2026-10-24 with the same `run_topk_inference` as
[all_misspredictions.ipynb](./all_misspredictions.ipynb). The job 4 model (`MViTv2_S_16x4_e`
chained asl100 -> asl300 -> asl1000 -> asl2000 on the 0-based plain split, Que_v2 job 18) was
stashed the same way on 2026-10-25. Its training set also has the three cutoff videos, which the
training gloss sets here leave out; 3 of 14,298 videos don't change the figures.

In [14]:
def headline(r: Run) -> dict[str, float]:
    table = load_table(r)
    wrong_exp = wrong_of(table)
    signer = permutation_test(wrong_exp, 'signer', signer_sets, n_perm)
    within = permutation_test(wrong_exp, 'signer', signer_sets, n_perm, strata='source')
    source = permutation_test(wrong_exp, 'source', source_sets, n_perm)
    is_related = [
        frozenset((g, p)) in cross or p in o
        for g, p, o in zip(wrong_exp['gloss'], wrong_exp['predicted'], wrong_exp['others'], strict=True)
    ]
    unrelated_exp = permutation_test(
        wrong_exp[~np.array(is_related)].reset_index(drop=True), 'signer', signer_sets, n_perm
    )
    no_copy_exp = table[~table['has_train_copy']]
    acc_by = no_copy_exp.groupby('signer_true')['top1'].mean() * 100
    feats, chose, avail = candidate_features(
        wrong_exp, list(wrong_exp['others']), classes, signer_sets, source_sets, train_counts, cross
    )
    odds = conditional_logit(feats, chose, avail, CANDIDATE_FEATURES).table()
    return {
        'top-1 (%)': table['top1'].mean() * 100,
        'wrong (no copy)': len(wrong_exp),
        'same signer (%)': signer.observed * 100,
        'shuffled (%)': signer.null_mean * 100,
        'p': signer.p,
        'shuffled within source (%)': within.null_mean * 100,
        'p within source': within.p,
        'unrelated: same signer (%)': unrelated_exp.observed * 100,
        'unrelated: shuffled (%)': unrelated_exp.null_mean * 100,
        'same source (%)': source.observed * 100,
        'source shuffled (%)': source.null_mean * 100,
        'same-signer OR': odds.loc['same signer', 'odds ratio'],
        'same-signer OR 95% low': odds.loc['same signer', 'OR 95% low'],
        'same-signer OR 95% high': odds.loc['same signer', 'OR 95% high'],
        'same-source OR': odds.loc['same source', 'odds ratio'],
        'same-source OR 95% low': odds.loc['same source', 'OR 95% low'],
        'same-source OR 95% high': odds.loc['same source', 'OR 95% high'],
        'top-1, signer has gloss (%)': acc_by[True],
        'top-1, signer lacks gloss (%)': acc_by[False],
    }


pd.DataFrame({format_exp_label(r.exp_no): headline(r) for r in [run, *other_runs]}).round(4)

,exp000,exp002,MViTv2_S_16x4_e_5mzfq88e_chained_s42
top-1 (%),38.8677,44.6336,53.0392
wrong (no copy),1613.0000,1447.0000,1205.0000
same signer (%),46.4352,36.4202,37.4274
shuffled (%),26.5482,25.0264,24.8540
p,0.0005,0.0005,0.0005
shuffled within source (%),38.8845,32.3774,33.5095
p within source,0.0005,0.0005,0.0005
unrelated: same signer (%),46.8171,35.7073,36.7532
unrelated: shuffled (%),25.9262,24.3804,23.9215
same source (%),71.6677,61.0919,60.1660


## Summary

Figures from this notebook, run 2026-10-24 (exp000 unless stated; wrong predictions exclude the
147 that are the gloss of an identical training copy, leaving 1,613 of 1,760).

* **The effect reproduces** on the 0-based labels, to the decimal: 46.4% of wrong predictions
  are a gloss the same signer performs in training, against 26.5% shuffled (ratio 1.75,
  p < 0.001); for source, 71.7% against 51.7%.
* **Signer, beyond source.** Shuffling only within source still leaves 46.4% against 38.9%
  (p < 0.001). In the conditional logit, with source, gloss popularity and two similar-sign
  measures held fixed, a gloss the signer performs is 2.46 times as likely to be the wrong
  prediction (95% CI 2.11-2.87), and one the source has (by another signer) 1.96 times
  (1.68-2.29). Both matter; the signer's is the larger.
* **Not similar signs.** Wrong predictions on the signer's glosses are no more often related to
  the true gloss (22.4%, by identical video or other-signer confusion) than the rest (23.6%).
  Among unrelated predictions alone: 46.8% against 25.9% (n = 1,241). So the wrong-prediction
  effect is evidence of an appearance (signer/recording) shortcut.
* **When right, it looks like the sign form.** Top-1 is 61.4% (n = 295) when the signer performs
  the true gloss in training against 38.3% (n = 2,428); the logistic odds ratio is 1.75
  (1.13-2.71, p = 0.012) with the gloss's training signers and count held fixed, and source adds
  nothing (p = 0.46). But the gain is only for the same variation: 65.2% (n = 253), against
  38.1% (n = 42) when the signer's training videos are another variation, which is no better
  than baseline. So the accuracy gain fits the signer's own form of the sign at least as well as
  appearance.
* **Spread across signers.** 97 signers have wrong predictions; signer 9 (`asl5200`, 383 wrong)
  gives 30% of the excess (260 hits against 163 expected), signers 9, 5, 4 and 11 give 48%, and
  15 signers give 81%. Without signer 9 the rate is 39.8% against 22.0%; without 9, 5 and 11,
  33.2% against 15.9% (ratio 2.09). Of signers expected at least one hit, only 20 and 41 fall
  short, so signer 20's *auction*-for-*before* is an example of the shortcut but not of a signer
  that drives it.
* **exp002** (44.6% top-1) shows the same pattern, weaker: 36.4% against 25.0% shuffled
  (32.4% within source), same-signer OR 1.63, same-source OR 1.53; top-1 58.6% against 45.4%.
* **The thesis's final asl2000 model** (job 4, `MViTv2_S_16x4_e` chained, 53.0% top-1; 1,205
  wrong predictions not explained by a copy) shows it too: 37.4% against 24.9% shuffled (33.5%
  within source, p < 0.001), 36.8% against 23.9% on unrelated glosses; same-signer OR 1.90
  (1.57-2.30), same-source OR 1.35 (1.13-1.62); top-1 65.1% against 54.6%. Across these three runs
  the effect is smaller in the more accurate models, but it doesn't go away.